# CBAM 注意力机制入门

前面学习 CNN 时，我们知道卷积会得到多个通道的特征图；学习 Swin Transformer 时，也接触过注意力。现在来看一种用于卷积特征的轻量注意力模块：CBAM。先理解它做什么、数据形状如何变化，再考虑代码实现。

## 1. 为什么需要 CBAM

一张特征图包含不同通道和不同空间位置的信息。面对某个任务，有些通道可能更有用，有些位置可能更值得关注。普通卷积能够学习特征，但输出的每个通道和位置不会因此自动拥有一个明确展示的、针对当前输入计算的权重。

CBAM 根据当前输入特征计算权重，对特征进行调整，让网络有机会突出有用信息、抑制相对不重要的信息。这是希望达到的效果，具体是否提升需要通过实验验证。

## 2. CBAM 是什么

CBAM 的全称是 Convolutional Block Attention Module，即卷积块注意力模块。它通常插入 CNN 的特征提取过程中，接收一个特征图，依次进行：

1. **通道注意力**：哪些通道更值得关注？
2. **空间注意力**：特征图的哪些位置更值得关注？

可以记成：输入特征 → 通道注意力 → 空间注意力 → 输出特征。原始 CBAM 采用先通道、后空间的顺序。它一般保持输入和输出的形状相同，方便与现有网络组合。

## 3. 先认识输入形状

假设输入特征为 $F$，形状为：

$$F \in \mathbb{R}^{B \times C \times H \times W}$$

- $B$：批次中的样本数。
- $C$：通道数，可以理解为不同类型的特征响应。
- $H, W$：特征图的高和宽。

例如 $B=2, C=64, H=W=28$，输入形状就是 $2 \times 64 \times 28 \times 28$。注意：通道不一定能简单对应到某个具体物体或语义。

## 4. 通道注意力：关注什么

通道注意力为每个样本的每个通道生成一个权重，同一通道内的所有空间位置共享该权重。

先分别对每个通道的 $H \times W$ 空间区域做**全局平均池化**和**全局最大池化**，得到两组通道描述，形状都是 $B \times C \times 1 \times 1$。两组描述分别经过同一个共享参数的多层感知机（MLP），结果相加，再通过 Sigmoid 得到通道权重：

$$M_c(F) = \sigma(\operatorname{MLP}(\operatorname{AvgPool}(F)) + \operatorname{MLP}(\operatorname{MaxPool}(F)))$$

其中 $\sigma$ 表示 Sigmoid；权重形状为 $B \times C \times 1 \times 1$。平均池化概括整体响应，最大池化保留最强响应；两者提供不同的概括信息。

把权重通过广播逐元素乘回原特征：

$$F' = M_c(F) \odot F$$

$F'$ 仍是 $B \times C \times H \times W$。例如某通道权重为 0.8，就将该通道各位置乘以 0.8。Sigmoid 权重介于 0 和 1 之间，因此这里的“突出”是相对于其他被抑制更多的特征而言。

## 5. 空间注意力：关注哪里

空间注意力接收通道调整后的 $F'$，为每个样本的每个空间位置生成一个权重，所有通道共享这个位置权重。

沿**通道维度**分别做平均池化和最大池化，得到两个 $B \times 1 \times H \times W$ 的图。将它们沿通道维度拼接，得到 $B \times 2 \times H \times W$；经过一个卷积（原始 CBAM 使用 $7 \times 7$ 卷积）和 Sigmoid，得到 $B \times 1 \times H \times W$ 的空间权重：

$$
M_s(F') = \sigma\left(f^{7 \times 7}\left([\operatorname{AvgPool}_c(F');\operatorname{MaxPool}_c(F')]\right)\right)
$$

其中方括号表示拼接，$f^{7 \times 7}$ 表示卷积。将权重广播后乘回 $F'$：

$$F'' = M_s(F') \odot F'$$

输出 $F''$ 的形状仍为 $B \times C \times H \times W$。空间权重较高的位置相对保留更多信息，但权重图不一定就是可靠的因果解释。

## 6. 用一个例子串起来

输入是 $2 \times 64 \times 28 \times 28$：

1. 通道池化和共享 MLP 得到 $2 \times 64 \times 1 \times 1$ 的通道权重。
2. 乘回输入，得到 $2 \times 64 \times 28 \times 28$。
3. 沿通道池化、拼接、卷积，得到 $2 \times 1 \times 28 \times 28$ 的空间权重。
4. 再乘回特征，输出仍是 $2 \times 64 \times 28 \times 28$。

两次池化的方向要区分：通道注意力压缩高和宽；空间注意力压缩通道。

## 7. 和 Transformer 注意力有什么区别

两者都通过输入相关的权重调整信息，但计算方式不同。CBAM 依次生成通道权重和空间权重，再与特征相乘；Transformer 自注意力通常通过 Query、Key、Value 计算 token 之间的信息交互。不要把两者当成同一个公式。

## 8. 小结与自测

CBAM 是对卷积特征进行重新加权的模块。先回答“关注什么通道”，再回答“关注哪些位置”；输出和输入的形状通常相同。

试着不看上文回答：

1. 输入为 $B \times C \times H \times W$ 时，两种注意力权重分别是什么形状？
2. 两个阶段分别沿哪些维度池化？为什么需要广播？
3. 平均池化和最大池化得到的信息有什么不同？
4. 空间注意力接收的是原始输入，还是通道调整后的特征？
5. CBAM 和 Transformer 自注意力的信息处理方式有何不同？